In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim

PROJECT_ROOT = "/content/drive/MyDrive/intel-image-classifier"
sys.path.insert(0, f"{PROJECT_ROOT}/src")
os.chdir(PROJECT_ROOT)
print("Connected to project folder.")

from preprocessing import get_transforms
from data_loader import get_loaders
from resnet_model import build_resnet18, unfreeze_for_finetuning, inspect_layer_status
from train import set_seed, train_one_epoch, validate, plot_curves

def main():
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dir = os.path.join(PROJECT_ROOT, "data/seg_train/seg_train")
    test_dir = os.path.join(PROJECT_ROOT, "data/seg_test/seg_test")

    for local_path in ["/content/data", "/content/intel_data"]:
        if not os.path.exists(train_dir) and os.path.exists(f"{local_path}/seg_train/seg_train"):
            train_dir = f"{local_path}/seg_train/seg_train"
            test_dir = f"{local_path}/seg_test/seg_test"

    os.makedirs("models", exist_ok=True)
    os.makedirs("outputs/plots", exist_ok=True)
    os.makedirs("outputs/metrics", exist_ok=True)

    # hyperparameters and dataloaders
    IMG_SIZE = 224
    BATCH_SIZE = 32
    STAGE1_EPOCHS = 6   # feature extraction
    STAGE2_EPOCHS = 8   # fine-tuning

    train_tf, val_tf = get_transforms(img_size=IMG_SIZE)
    train_loader, val_loader, _, class_names = get_loaders(
        train_dir, test_dir, train_tf, val_tf, batch_size=BATCH_SIZE
    )

    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    start_time = time.time()

    # feature extraction
    model = build_resnet18(num_classes=len(class_names), freeze_backbone=True).to(device)
    inspect_layer_status(model)

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, STAGE1_EPOCHS + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc = validate(model, val_loader, criterion, device)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        print(f"[Stage 1 | Epoch {epoch:02d}/{STAGE1_EPOCHS:02d}] "
              f"Train Loss: {t_loss:.4f}, Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f}, Acc: {v_acc:.4f}")

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save({
                'stage': 'feature_extraction',
                'model_state_dict': model.state_dict(),
                'class_names': class_names,
                'img_size': IMG_SIZE
            }, "models/resnet18_best.pth")

    # fine-tuning
    unfreeze_for_finetuning(model, unfreeze_layer3=False)
    inspect_layer_status(model)

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)

    for epoch in range(1, STAGE2_EPOCHS + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc = validate(model, val_loader, criterion, device)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        print(f"[Stage 2 | Epoch {epoch:02d}/{STAGE2_EPOCHS:02d}] "
              f"Train Loss: {t_loss:.4f}, Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f}, Acc: {v_acc:.4f}")

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save({
                'stage': 'fine_tuning',
                'model_state_dict': model.state_dict(),
                'class_names': class_names,
                'img_size': IMG_SIZE
            }, "models/resnet18_best.pth")

    elapsed = time.time() - start_time
    print(f"\nResNet-18 training finished in {elapsed:.1f}s. Best Validation Accuracy: {best_val_acc:.4f}")

    with open("outputs/metrics/resnet18_history.json", "w") as f:
        json.dump(history, f, indent=4)
    plot_curves(history, "outputs/plots/resnet18_training_curves.png")

if __name__ == "__main__":
    main()